=============================================================================
ORIGIN DESTINATION MATRIX HEATMAP
Kitsap Transit — StreetLight Data
=============================================================================
Straight OD data output (ignore middle filters) with identical zones
as both origins and destinations.
=============================================================================

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

=============================================================================
CONFIGURATION
=============================================================================

In [ ]:
INPUT_FILE = r'C:\Users\rebeccasc\Documents\Scripts\BI_commerce_TDM\2014392_BI_OD_COMM_2024\2014392_BI_OD_COMM_2024_od_trip_all.csv'
OUTPUT_DIR = r'C:\Users\rebeccasc\Documents\Scripts\BI_commerce_TDM\od_matrix'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
VOLUME_COL   = 'Average Daily O-D Traffic (StL Volume)'
ORIGIN_COL   = 'Origin Zone Name'
DEST_COL     = 'Destination Zone Name'

In [ ]:
# Zone display order — edit to match your actual zone names exactly
ZONE_ORDER = [
    #'FERRY_TERMINAL',
    #'WINSLOW',
    #'COPPERTOP',
    #'LYNWOOD_CENTER',
    #'MID_ISLAND',
    #'NORTH_ISLAND',
    #'SOUTH_ISLAND'

    'SR305_S',
    'WINSLOW_WAY',
    'LYNWOODCENTER',
    'HIGH_SCHOOL_RD',
    'SR305_MID',
    'SPORTSMAN_CLUB',
    'SR305_N'
]

In [ ]:
# Short display labels for the axes
ZONE_LABELS = {
    #'FERRY_TERMINAL': 'Ferry Terminal',
    #'WINSLOW':        'Winslow',
    #'COPPERTOP':      'Coppertop',
    #'LYNWOOD_CENTER': 'Lynwood Ctr',
    #'MID_ISLAND':     'Mid Island',
    #'NORTH_ISLAND':   'North Island',
    #'SOUTH_ISLAND':   'South Island'
    'HIGH_SCHOOL_RD': 'High School Rd',
    'SPORTSMAN_CLUB': 'Coppertop',
    'SR305_MID': '305 midway',
    'SR305_N': '305 off-island',
    'SR305_S': 'Ferry Terminal',
    'LYNWOODCENTER': 'Lynwood Center',
    'WINSLOW_WAY': 'Winslow'

In [ ]:
}

In [ ]:
DAY_TYPE_ORDER = [
    '0: All Days (M-Su)',
    '1: Monday (M-M)',
    '2: Tuesday (Tu-Tu)',
    '3: Wednesday (W-W)',
    '4: Thursday (Th-Th)',
    '5: Friday (F-F)',
    '6: Saturday (Sa-Sa)',
    '7: Sunday (Su-Su)'
]

In [ ]:
TIME_PERIODS = {
    'All Day':    '0: All Day (12am-12am)',
    'Morning':    'Peak AM',
    'Midday':     'Mid-Day',
    'Evening':    'Peak PM',
    'Late PM':    'Late PM'
}

In [ ]:
def extract_hour(day_part_str):
    try:
        s = str(day_part_str)
        if 'all day' in s.lower():
            return None
        idx = int(s.split(':')[0].strip())
        return idx - 1
    except:
        return None

In [ ]:
def get_day_category(day_type_str):
    s = str(day_type_str).lower()
    if 'sunday'    in s: return 'sunday'
    if 'saturday'  in s: return 'saturday'
    if 'monday'    in s: return 'monday'
    if 'tuesday'   in s: return 'tuesday'
    if 'wednesday' in s: return 'wednesday'
    if 'thursday'  in s: return 'thursday'
    if 'friday'    in s: return 'friday'
    return 'all'
# =============================================================================
# LOAD DATA
# =============================================================================

In [ ]:
print("=" * 70)
print("BAINBRIDGE ISLAND OD MATRIX")
print("=" * 70)
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

In [ ]:
try:
    df = pd.read_csv(INPUT_FILE)
    print(f"Loaded {len(df):,} records.")
except Exception as e:
    print(f"ERROR: {e}")
    exit(1)

In [ ]:
df['Hour']         = df['Day Part'].apply(extract_hour)
df['Day_Category'] = df['Day Type'].apply(get_day_category)

In [ ]:
# Validate zones
found_origins = df[ORIGIN_COL].unique().tolist()
found_dests   = df[DEST_COL].unique().tolist()
missing = [z for z in ZONE_ORDER if z not in found_origins and z not in found_dests]
if missing:
    print(f"WARNING — zones not found in data: {', '.join(missing)}")
    print("  Update ZONE_ORDER to match your actual zone names.\n")

In [ ]:
present_zones = [z for z in ZONE_ORDER if z in found_origins or z in found_dests]

=============================================================================
BUILD OD MATRIX FUNCTION
=============================================================================

In [ ]:
def build_od_matrix(data, zones, zone_labels):
    """Pivot to origin x destination matrix, reindex to zone order."""
    pivot = data.pivot_table(
        values=VOLUME_COL,
        index=ORIGIN_COL,
        columns=DEST_COL,
        aggfunc='sum',
        fill_value=0
    )
    # Reindex to zone order, keeping only present zones
    z = [z for z in zones if z in pivot.index or z in pivot.columns]
    pivot = pivot.reindex(index=z, columns=z, fill_value=0)
    pivot.index   = [zone_labels.get(i, i) for i in pivot.index]
    pivot.columns = [zone_labels.get(c, c) for c in pivot.columns]
    return pivot

In [ ]:
def plot_od_matrix(pivot, title, outpath, figsize=(12, 10)):
    """Render and save an OD matrix heatmap."""
    fig, ax = plt.subplots(figsize=figsize)
    max_val = pivot.values.max()

    sns.heatmap(
        pivot,
        annot=True,
        fmt='.0f',
        cmap='Blues',
        linewidths=0.5,
        vmin=0,
        vmax=max_val,
        cbar_kws={'label': 'Avg Daily Vehicle Trips'},
        ax=ax,
        annot_kws={'size': 9}
    )

    # Highlight diagonal (intra-zone trips)
    for i in range(len(pivot)):
        ax.add_patch(plt.Rectangle((i, i), 1, 1,
                     fill=False, edgecolor='#e74c3c',
                     linewidth=2, clip_on=False))

    ax.set_title(title, fontsize=12, pad=15)
    ax.set_xlabel('Destination', fontsize=10)
    ax.set_ylabel('Origin', fontsize=10)
    ax.tick_params(axis='x', rotation=45, labelsize=9)
    ax.tick_params(axis='y', rotation=0,  labelsize=9)
    plt.tight_layout()
    plt.savefig(outpath, dpi=300, bbox_inches='tight')
    plt.close()

=============================================================================
ANALYSIS 1: ALL DAYS, ALL HOURS
=============================================================================

In [ ]:
print("Generating OD matrices...\n")

In [ ]:
all_days = df[df['Day Part'].str.contains('All Day', case=False, na=False)]
pivot_all = build_od_matrix(all_days, present_zones, ZONE_LABELS)

In [ ]:
plot_od_matrix(
    pivot=pivot_all,
    title='OD Trip Matrix — All Days, All Hours\nBainbridge Island Commercial Zones\n(Red outline = intra-zone trips)',
    outpath=os.path.join(OUTPUT_DIR, 'OD_Matrix_AllDays.pdf')
)
print("  All Days matrix saved.")

=============================================================================
ANALYSIS 2: BY DAY TYPE
=============================================================================

In [ ]:
day_type_map = {
    'Monday':    ['monday'],
    'Tuesday':   ['tuesday'],
    'Wednesday': ['wednesday'],
    'Thursday':  ['thursday'],
    'Friday':    ['friday'],
    'Saturday':  ['saturday'],
    'Sunday':    ['sunday']
}

In [ ]:
for day_label, day_cats in day_type_map.items():
    subset = df[
        (df['Day_Category'].isin(day_cats)) &
        (df['Hour'].notna())
    ]
    if subset.empty:
        print(f"  {day_label}: no data — skipped")
        continue

    pivot = build_od_matrix(subset, present_zones, ZONE_LABELS)
    plot_od_matrix(
        pivot=pivot,
        title=f'OD Trip Matrix — {day_label}\nBainbridge Island Commercial Zones\n(Red outline = intra-zone trips)',
        outpath=os.path.join(OUTPUT_DIR, f'OD_Matrix_{day_label}.pdf')
    )
    print(f"  {day_label} matrix saved.")

=============================================================================
ANALYSIS 3: TIME OF DAY PANELS — ALL ON ONE PAGE
=============================================================================

In [ ]:
hour_ranges = {
    'Morning (6a–10a)':      (6,  10),
    'Midday (10a–3p)':       (10, 15),
    'Early Evening (3p–7p)': (15, 19),
    'Late Evening (7p–12a)': (19, 24)
}

In [ ]:
df_hourly = df[df['Hour'].notna()].copy()

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(20, 18))
axes = axes.flatten()

In [ ]:
for ax_idx, (period_label, (h_start, h_end)) in enumerate(hour_ranges.items()):
    subset = df_hourly[
        (df_hourly['Hour'] >= h_start) &
        (df_hourly['Hour'] < h_end)
    ]
    if subset.empty:
        axes[ax_idx].set_visible(False)
        continue

    pivot = build_od_matrix(subset, present_zones, ZONE_LABELS)
    max_val = pivot.values.max()

    sns.heatmap(
        pivot,
        annot=True, fmt='.0f',
        cmap='Blues',
        linewidths=0.5,
        vmin=0, vmax=max_val if max_val > 0 else 1,
        cbar_kws={'label': 'Avg Daily Trips'},
        ax=axes[ax_idx],
        annot_kws={'size': 8}
    )
    for i in range(len(pivot)):
        axes[ax_idx].add_patch(plt.Rectangle((i, i), 1, 1,
                               fill=False, edgecolor='#e74c3c',
                               linewidth=2, clip_on=False))

    axes[ax_idx].set_title(period_label, fontsize=11, fontweight='bold')
    axes[ax_idx].set_xlabel('Destination', fontsize=9)
    axes[ax_idx].set_ylabel('Origin', fontsize=9)
    axes[ax_idx].tick_params(axis='x', rotation=45, labelsize=8)
    axes[ax_idx].tick_params(axis='y', rotation=0,  labelsize=8)

In [ ]:
fig.suptitle(
    'OD Trip Matrix by Time of Day — Bainbridge Island Commercial Zones\n'
    'All Weekdays   Red outline = intra-zone trips',
    fontsize=13, y=1.01
)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'OD_Matrix_TimeOfDay.pdf'),
            dpi=300, bbox_inches='tight')
plt.close()
print("  Time-of-day matrix saved.")

=============================================================================
EXPORT SUMMARY TABLE
=============================================================================

In [ ]:
pivot_all.to_csv(os.path.join(OUTPUT_DIR, 'OD_Matrix_AllDays.csv'))
print("\n  Summary CSV saved.")
print(f"\nAll outputs in: {OUTPUT_DIR}")
print(f"Complete: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")